# Laboratorio 10 · Explainable AI · Signet C4
**Diego Rosales — 23258 · Juan Francisco Martínez — 23617**

Se aplica la oclusión al checkpoint final **C4 sin reentrenar**. Las tablas y figuras se generan ejecutando este notebook contra audio real de test. Los auxiliares de esta carpeta forman parte del proceso reproducible.

**Decisión y alcance:** el equipo indicó usar C4 y proporcionó [sus pesos](https://huggingface.co/r0sV/signet); la ficha lo identifica como modelo servido. C4 se entrenó sobre ASVspoof 2019 + español moderno (FLEURS, Edge, F5 y OmniVoice), pero el equipo confirmó que los audios españoles solo están en la computadora del compañero. Esta ejecución explica **ASVspoof 2019 LA eval**: cinco casos y una muestra estratificada de 28. No se presentan explicaciones del español; las sondas G.711/GSM pareadas sobre cinco casos no sustituyen la evaluación completa de códecs. El README documenta cómo ampliar a esos splits cuando se obtengan los originales.

Ejecutar desde kernel limpio y en orden, desde la raíz: `uv run --no-sync python -m lab10.run`. Antes: adquirir C4 y el audio con los comandos del README. Las variables `LAB10_*` se leen al inicio; sus valores efectivos se exportan. No se versionan pesos, audio ni credenciales.

In [ ]:
# Se localiza la raíz tanto al lanzar desde ella como desde Jupyter en lab10/.
import os
import sys
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "protocols").is_dir())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import json
import time

import pandas as pd
from IPython.display import Image, display

from lab10.audit import OUT, repository_audit
from lab10.pipeline import (
    Predictor,
    benchmark,
    explain_global,
    explain_local,
    export_parameters,
    fidelity,
    figures,
    measured_findings,
    methods_table,
    prepare,
    read_wave,
    reproduce,
    risk_probes,
    sample_cases,
    seed_everything,
    settings_from_env,
)

s = settings_from_env()
hardware = seed_everything(s)
display(pd.Series(hardware, name="Entorno real"))
print("Modelo solicitado:", s.model or "PENDIENTE de confirmar")
print("Splits:", s.splits, "Semilla:", s.seed, "Lote:", s.batch)

from lab10.interpretation import deployment_verification, export_decision_tables
from lab10.pipeline import precision_audit


## 1. Auditoría: versión, datos y score
Se comparan **C2, C3 y C4**. El README raíz se actualizó a C4 con revisión del Hub, hashes y configuración concretos. PRODUCT/Docker conservan configuración histórica C2, identificada como tal, y no se confunde con el artefacto final publicado. **Se utiliza C4 por indicación expresa del equipo y porque su ficha de Hugging Face lo identifica como servido**, adaptado a FLEURS + Edge/F5/OmniVoice. La elección no se basa solo en la fecha: mejora EER en español (24.18% → 3.99% frente a C2), con degradación en ASVspoof limpio y con códec. La presentación final no está versionada; no consta su revisión. `provenance.json` registra revisión del Hub, SHA256 de pesos, época, datos, scores y código.

**Clase positiva:** 1 = sintética, 0 = genuina. El score de CSV es softmax crudo. La API decide con **`C4/thresholds.json`: t1=0.957 y t2=0.9978**. El archivo `value_thresholds.json` tiene valores sin redondear ligeramente distintos; se utilizan los de la API para explicar sus acciones efectivas. Entre t1 y t2 hay verificación. El umbral EER no se usa como umbral operativo. C2 tiene t1=t2, pero sus umbrales no se trasladan a C4.

La calibración de Platt de C4 (ajustada en `mix_dev`) transforma logit(score) mediante `a=1.0293036053`, `b=-3.0592936052` y el prior configurado; el panel muestra esa probabilidad, pero **la acción depende del score crudo**. No se denomina confianza a los log-odds. La unidad de este análisis es una ventana de evaluación, no toda la llamada.


**Referencia servida exacta:** `r0sV/signet@fa67b890666c8f92477a27d4155032afbf86b405/C4/model.onnx`, SHA256 `a7e03a0410e9685e3cca6a3624910bc9ee9f15929bc1cc8d1a81d0b201c3750b`. Se verifica ese ONNX con `app.inference.ServedModel` y los mismos 33 originales. No es una consulta a un servicio remoto ni equivalencia probada de todas las perturbaciones. Los scores GPU/bf16 son históricos de CSV, no una nueva ejecución GPU.

In [ ]:
comparison, audit = repository_audit()
display(comparison[["run", "split", "n", "eer", "checkpoint_exists", "training_commit"]])
print(json.dumps(audit, ensure_ascii=False, indent=2))


## 2. Investigación y elección del método
La audiencia es el **agente bancario** y el **analista de fraude**. Se selecciona la oclusión temporal y por bandas porque pregunta qué parte del audio cambia la evidencia del detector y funciona con XLS-R congelado, cuyo `extract()` usa `torch.no_grad()`. IG necesitaría habilitar gradientes de entrada a través del extractor sin entrenar ni cambiar pesos; tiene más costo de memoria y una línea base acústica discutible. Grad-CAM de LFCC+LCNN explica el modelo A: no se presenta como explicación del modelo C.

Una regla transparente (árbol pequeño o regresión logística sobre rasgos acústicos) sería **interpretable por diseño**; una oclusión es **posterior** a una red opaca. No se dispone de una evaluación de un baseline interpretable; por tanto, no se presenta una comparación de rendimiento no medida ni se entrena uno para este análisis. Si alcanzara rendimiento comparable en este escenario de alto impacto, habría razones para preferirlo.

Fuentes primarias consultadas: [Captum: oclusión](https://captum.ai/api/occlusion.html), [Captum: IG](https://captum.ai/api/integrated_gradients.html), [Selvaraju et al.: Grad-CAM](https://arxiv.org/abs/1610.02391). Los costos de alternativas son aproximaciones por número de forwards/backwards, no benchmarks medidos.

In [ ]:
methods = methods_table()
display(methods)


## 3. Propiedad y parámetros de diseño
Se definen $s(x)=z_1(x)-z_0(x)=\log(P_1/P_0)$ y $\Delta_i=s(x)-s(x\text{ con región }i\text{ retirada})$.

- $\Delta_i>0$: retirar baja la evidencia sintética. $\Delta_i<0$: retirar la aumenta; en un genuino puede representar eliminación de evidencia de naturalidad. No se denomina confianza a log-odds.
- Se mide **sensibilidad a una perturbación**: no causalidad, no aditividad, no garantía de independencia entre tramos/bandas. No se suman los efectos de ventanas solapadas.
- Para **IG** con una función diferenciable, línea base $x'$ y suficiente integración, $\sum_j IG_j(x;x')\approx s(x)-s(x')$. La completitud es relativa a la línea base y tiene error numérico; no es una propiedad de estos mapas de oclusión. IG no se implementa ni se atribuyen sus garantías a este método.

Se mantiene el ancho temporal de **0.25 s** y se documenta el paso de **0.125 s**. A diferencia del código inicial, se añade una última ventana de ancho completo alineada al final para cubrir las 64,600 muestras. Se mantienen seis bandas **0–300, 300–1000, 1000–2000, 2000–3400, 3400–5000 y 5000–8000 Hz**; la última incluye Nyquist, que el operador `< hi` original dejaba fuera. El reemplazo consiste en silencio o anulación de una banda mediante FFT. La FFT rectangular y los saltos al silencio pueden crear artefactos: se miden energía, saltos y clipping, y se comparan máscaras duras con rampas de **10 ms**.

No se modifican pesos, calibración ni umbrales usando test. La semilla **42** fija random/NumPy/PyTorch, selección global y controles de fidelidad. Float32 es el valor por defecto; bfloat16 CUDA es opcional y queda registrado. `deterministic_algorithms(warn_only=True)` registra límites prácticos: otra versión de kernel/GPU puede variar numéricamente.

**Perturbador principal:** las Figuras 1 y 2 usan silencio **brusco** en tiempo y banda FFT anulada en frecuencia. La rampa de 10 ms se aplica únicamente al control de sensibilidad de la Figura 3. El signo de Δ se refiere siempre a evidencia sintética: en un genuino, Δ negativo puede indicar pérdida de evidencia de naturalidad; no convierte la señal en prueba de identidad. El cambio medio top al introducir rampa se calcula, no se supone pequeño.

## 4. Plan mínimo de datos (sin cargar pesos)
Se seleccionan **cinco IDs únicos**: acierto con score extremo y correcto, error, menor distancia a **t2** (umbral de transferencia), subgrupo (español/códec; si solo se utiliza ASVspoof original, ataque **A10**) y un genuino distinto. La inclusión es dirigida, descriptiva, no una estimación de prevalencia ni una forma de ajustar test.

La muestra global usa hasta **dos audios por estrato split × clase × ataque × códec**, semilla 42. No se selecciona por disponibilidad ni se llama global al ejemplo local. `required_audio.csv` permite descargar **solo la unión de ambas muestras**. Para combinar telefonía y original, recordar que los códecs provienen de los mismos IDs de eval y no son réplicas independientes.

Este plan puede generarse antes de conseguir los pesos. C4 y sus umbrales efectivos quedan fijados por defecto; los archivos se adquieren antes del notebook.

La comparación entre clases es **exploratoria**: 2 genuinos frente a 26 sintéticos (2 por cada ataque A07–A19). Dos genuinos no permiten caracterizar la variabilidad poblacional de su clase. La muestra se mantiene y esta mezcla no se generaliza al banco. El caso 4 es **A10 sin códec telefónico añadido**: comparte ataque con el falso negativo que continúa del caso 2, lo que permite contrastar resultados operativos dentro de un mismo ataque.

In [ ]:
from lab10.plan_data import plan_data

if s.model in {"C2", "C3", "C4"} and Path(s.thresholds).is_file():
    planned_local, planned_global = plan_data(s)
    display(planned_local[["utt_id", "split", "label", "cached_score", "reason"]])
    display(pd.read_csv(OUT / "global_distribution.csv"))
else:
    print("Plan pendiente de modelo final y umbrales operativos propios.")


## 5. Carga y verificación obligatoria
A partir de aquí se exigen **checkpoint entrenado y audio real**. Se reutilizan `src.models.build_model` y `AudioDataset(train=False)`, con `eval()` e inferencia sin gradientes. Se construye la arquitectura XLS-R desde su config oficial versionada en `assets/xlsr_config.json` y se cargan **todos** los tensores del checkpoint con `strict=True, assign=True` y `mmap=True`. Así se evita descargar un segundo backbone o duplicar 1.26 GB en RAM; no hay fallback a pesos aleatorios ni entrenamiento. La configuración oficial proviene de `facebook/wav2vec2-xls-r-300m`, revisión `1a640f32ac3e39899438a2931f9924c02f080a54`.

Los protocolos se contrastan con los CSV: IDs únicos, etiquetas iguales y cobertura completa de scores; se verifica la separación de entrenamiento por ID/ruta y se informa el solapamiento de hablantes cuando existe metadato. El español tiene metadatos de hablante incompletos: no puede certificarse la ausencia de fuga por identidad. La época del checkpoint debe corresponder a las métricas guardadas. Se comparan los scores recalculados con tolerancia absoluta **0.005** y se registran todos los desacuerdos, sin ampliar esa tolerancia. La ejecución histórica GPU/bfloat16 no se reproduce exactamente en CPU: se contrasta también CPU/bfloat16 para distinguir identidad del checkpoint de equivalencia numérica. El SHA256 LFS y la época identifican C4; las explicaciones proceden de CPU/PyTorch float32; se comprueba por separado su correspondencia numérica con el ONNX publicado mediante la clase real de la API. `score_agreement.json` y `precision_audit.csv` conservan diferencias y cambios de acción. No se atribuye toda la deriva a una única causa sin el entorno original. El caso cercano se eligió por mínimo histórico, no se afirma que sea el mínimo actual de los 71,237 audios. Este contraste es empírico; los CSV antiguos no tienen hash del checkpoint.

**Unidad explicada:** primeras **64,600 muestras / 16,000 Hz = 4.0375 s**; mono, resample del dataset, clamp [-1,1], repetición de cortos y recorte inicial. El panel, en cambio, analiza hasta 20 s y promedia logit(score) de ventanas con voz; si ninguna tiene voz, todas votan. Aquí los scores/acciones son **de la ventana de evaluación**, no de una llamada multiventana. No se atribuye una decisión de llamada completa a un solo mapa.

In [ ]:
model, cfg, datasets, pool, provenance = prepare(s)
predict = Predictor(model, s)
local_plan, global_sample = sample_cases(pool, s, provenance["thresholds"])
print("Checkpoint SHA256:", provenance["checkpoint_sha256"])
display(pd.read_csv(OUT / "split_integrity.csv"))
display(pd.read_csv(OUT / "data_inventory.csv"))


In [ ]:
# Benchmark sincronizado: cinco predicciones tras calentamiento, B=1.
# No incluye lectura de audio ni carga del checkpoint; no equivale a ONNX en producción.
first_wav = read_wave(local_plan.iloc[0], datasets)
prediction_times = benchmark(predict, first_wav)
stage_costs = []
def measured_stage(name, function, *args):
    predict.sync()
    start, calls, inputs = time.perf_counter(), predict.calls, predict.inputs
    result = function(*args)
    predict.sync()
    stage_costs.append({"kind": name, "seconds": time.perf_counter()-start,
                        "forwards": predict.calls-calls, "inputs": predict.inputs-inputs})
    return result
local, verified, waves = measured_stage(
    "score_verification", reproduce, predict, datasets, local_plan, global_sample,
    provenance["thresholds"])
display(local[["utt_id", "split", "label", "prediction", "score", "action", "t1", "t2", "reason"]])
display(verified[["utt_id", "cached_score", "score", "score_abs_diff", "cached_action", "action"]])

precision_audit(predict, verified, waves)
display(pd.Series(json.loads((OUT / "score_agreement.json").read_text())))
display(pd.read_csv(OUT / "precision_audit.csv"))

# Contraste empírico del ONNX publicado: no se infiere equivalencia por usar float32.
deployment, runtime_changes, deployment_summary = deployment_verification(verified, waves, s)
display(runtime_changes)
display(pd.Series(deployment_summary))


## 6. Explicaciones locales y control de perturbaciones
Se generan **32 posiciones temporales** (0.25 s, paso 0.125 s y ajuste final) y seis bandas por ID. El score base ya se verificó. Se conserva el signo de todas las diferencias. En genuinos, quitar una señal de naturalidad puede elevar s(x), produciendo Δ negativo.

La comparación de fidelidad retira **tres regiones no solapadas**, 0.75 s por caso: las de mayor evidencia orientada a la clase predicha frente a **cinco controles aleatorios** con la misma duración. Para la clase 0 el signo se orienta como −Δ. Ambas variantes se evalúan con máscara dura y rampa de 10 ms. Es una evaluación descriptiva ligada al perturbador, **no una prueba causal ni independiente** de la explicación. Elegir top por el mismo perturbador favorece sus resultados; el control suavizado examina parte de esa dependencia.

No se guarda audio alterado en Git. `perturbation_qc.csv` compara RMS, energía retirada, salto entre muestras y clipping. La figura muestra la forma original, el efecto temporal y el efecto espectral, con escala y unidades.

La clasificación binaria usa el corte t2. **0 binario no significa continuar**: entre t1 y t2 la acción es verificar. El caso 2 es el falso negativo que deja continuar una voz sintética; los casos 3 y 4 son negativos en t2 que **sí activan revisión**. La orientación de fidelidad en clase 0 es la del corte binario, no una etiqueta operacional de seguridad.

In [ ]:
temporal, local_bands, local_times, qc = measured_stage(
    "local_occlusion", explain_local, predict, local, waves)
fidelity_frame = measured_stage("fidelity", fidelity, predict, local, waves, temporal)
probes = measured_stage("risk_probes", risk_probes, predict, local, waves, provenance["thresholds"])
display(qc.groupby("kind")[["relative_removed_energy", "max_adjacent_jump", "clipped_fraction"]].agg(["mean", "max"]))
display(fidelity_frame)


## 7. Vista global y análisis de errores
Se agrega la muestra definida antes de leer audio, sin sustituir archivos faltantes. Se informa **N real** y distribución por dominio/clase/ataque/códec. Se exporta cada Δ, media con signo, media |Δ|, desviación y fracción positiva: no se cancela evidencia de signos opuestos silenciosamente. Las medias estratificadas describen esta muestra, **no** la mezcla natural del banco.

La banda 300–3400 Hz es relevante para telefonía; los rangos exteriores pueden perderse por el canal. Se examina la sensibilidad por dominio y códec **si esos splits están disponibles**. Si solo se usa ASVspoof original, no se considera validado el desempeño en códecs reales ni en español. Las sondas de banda telefónica no son un códec real y no sustituyen esa validación.

Se comparan errores y aciertos, y se mide la sensibilidad al volumen, banda y silencio inicial. Ni la saliencia ni un solapamiento cero por ID prueban ausencia de fuga. Para comprobar un proxy de idioma/códec harían falta pares del mismo hablante/contenido en ambos canales y metadatos completos de hablantes/generadores. Una perturbación que cambie la predicción no demuestra que corregiría la llamada original.

También se construyen **pares G.711 μ-law y GSM** de los cinco casos con `scripts.precompute_codecs.apply_codec`, ffmpeg, retorno a 16 kHz y recorte/repetición a 64,600 muestras. Son sondas nuevas del mismo test, sin sobrescribir protocolos ni atribuirles los scores históricos de `asv19_eval_codec`. Se registran score/acción y versión de ffmpeg; cinco casos dirigidos no estiman robustez poblacional.

Las seis sondas se aplican a cada uno de los cinco casos dirigidos (no son 30 llamadas independientes). Se exportan `probe_action_changes.csv` y `probe_directions.csv` para identificar casos, intervención y dirección. Los cambios G.711/GSM y los de volumen/banda/silencio son señales para investigar; no estiman una tasa de fallo. La vista global separa n de cada clase y se rotula exploratoria.

In [ ]:
global_attr, global_cost = measured_stage(
    "global_bands", explain_global, predict, global_sample, verified, waves)
case_table, probe_changes, sampling_summary, fidelity_control = export_decision_tables(
    local, global_sample, probes, fidelity_frame, provenance["thresholds"])
figures(local, waves, temporal, local_bands, global_attr, fidelity_frame)
findings = measured_findings(local, global_attr, fidelity_frame, local_times,
                             prediction_times, global_cost)
parameters = export_parameters(s, provenance, local, global_sample)
pd.DataFrame(stage_costs).to_csv(OUT / "stage_costs.csv", index=False)
display(pd.read_csv(OUT / "global_summary.csv"))
display(pd.read_csv(OUT / "global_distribution.csv"))
display(pd.read_csv(OUT / "stage_costs.csv"))
display(pd.Series(findings))
for filename in ["local_explanations.png", "global_explanations.png", "fidelity.png"]:
    display(Image(filename=str(OUT / filename)))

# Lectura automática del resultado medido, sin escribir cifras manualmente en paralelo.
from IPython.display import Markdown

codec_checks = probes[probes.probe.str.startswith("codec_")]
codec_changes = int((codec_checks.action != codec_checks.base_action).sum())
error_row = local[local.reason == "error"].iloc[0]
display(Markdown(
    f"**Lectura de la ejecución:** el error `{error_row.utt_id}` ({error_row.attack}, "
    f"sintético) recibe score {error_row.score:.6f} y acción **{error_row.action}**. "
    f"Top-3 supera al control en {findings['fidelity_top_beats_control']}/5 casos; "
    f"la rampa modifica ese efecto en {findings['ramp_mean_absolute_change']:.2f} "
    f"log-odds (media absoluta). Los pares G.711/GSM cambian acción en "
    f"{codec_changes}/{len(codec_checks)} sondas. Estas son sensibilidades medidas "
    f"sobre casos dirigidos, sin prueba causal ni estimación de prevalencia. "
    f"Se conserva la oclusión para auditoría; no se dispone de evidencia XAI del test español original."))

display(case_table)
display(probe_changes)
display(pd.Series(sampling_summary))
display(fidelity_control)


## 8. Interpretación, ética y uso
Las lecturas cuantitativas proceden de `findings.json`, `risk_probes.csv` y las figuras anteriores, **no de resultados esperados**. La selección debe revisarse después de reproducir los scores: un error candidato que deje de ser error bloquea el informe. Los cambios de acción respecto al CSV quedan visibles.

La interfaz del agente mostraría **la acción y la calidad del audio**; el analista podría desplegar tramos y bandas sensibles con el signo y score original/perturbado. Una región saliente no constituye prueba de fraude. Ante inestabilidad de canal o voz insuficiente, se recomienda solicitar otro factor y revisión humana; los umbrales no se modifican con test.

Se selecciona la oclusión por compatibilidad y comunicación; se conservaría su uso para **auditoría y lotes**, sujeto al costo efectivamente medido y a su dependencia del reemplazo. Los mapas revelan localización y signo de sensibilidad que EER/AUC no muestran. La robustez esperada no se considera confirmada sin pruebas emparejadas. El benchmark usa el hardware/precisión actuales; sus latencias no se extrapolan a otra CPU, GPU o ONNX.

La voz es biométrica; no se publican audios privados. Persisten riesgos de acento/edad no medidos, sintetizadores nuevos y evasión adversarial. No usar como prueba de identidad, único factor de bloqueo/autorización, ni en idiomas, códecs o ataques físicos no validados. Si un baseline interpretable alcanzara rendimiento comparable, el impacto bancario favorecería esa alternativa, pero hoy no se dispone de esa medición.

**Hallazgo operativo central:** los dos cambios GPU histórico→CPU se identifican en `runtime_action_changes.csv` con acción y score en GPU/bf16 histórico, CPU/PyTorch y CPU/ONNX publicado. La conclusión sobre despliegue se basa en esa comprobación, no en una supuesta equivalencia por precisión. Un score inestable cerca del umbral puede cambiar el control aplicado aunque las métricas agregadas sean buenas. La dependencia de la rampa limita la estabilidad de magnitudes/localizaciones; se conserva la oclusión como auditoría, no explicación causal estable.